In [ ]:
import numpy as np
import torch
import pprint
import onnx
import copy
import hls4ml
from torch.utils.data import DataLoader
from utils.utils import read_object
from hls4ml.model.profiling import numerical
from onnx import numpy_helper
from cnn_base import t1_1_quantizedCNN
from sklearn.metrics import accuracy_score
%matplotlib inline
torch.manual_seed(4)

def print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key), end='')
        if isinstance(value, dict):
            print()
            print_dict(value, indent + 1)
        else:
            print(':' + ' ' * (20 - len(key) - 2 * indent) + str(value))

def trace_layers(config):
    config['Model']['TraceOutput'] = True
    for layer in config['LayerName']:
        config['LayerName'][layer]['Trace'] = True


DEBUG = False

In [ ]:
onnx_model_path = "./hls4ml_onnxs/sat6-cnn-t1w8_quant.onnx"
pytorch_model_path = "./pytorch_models/sat6-cnn-t1w8_quant.pt"
batch_size = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_dataset_output_name = "./dataset/test_dataset.pkl"

dataset_test = read_object(test_dataset_output_name)
loader_test = DataLoader(dataset_test, batch_size, shuffle=False)

X_test = dataset_test.tensors[0].numpy().astype(np.float32)
y_test = dataset_test.tensors[1].numpy().astype(np.int32)


if DEBUG:
    onnx_model = onnx.load(onnx_model_path)

    print("=== Nós do grafo (nome, tipo) ===")
    for node in onnx_model.graph.node:
        print(f"{node.name:25s} -> {node.op_type}")

    print("\n=== Input ===")
    for inp in onnx_model.graph.input:
        print(inp.name, [d.dim_value for d in inp.type.tensor_type.shape.dim])

In [ ]:
onnx_model = onnx.load(onnx_model_path)

config = hls4ml.utils.config_from_onnx_model(
    onnx_model, granularity='name', backend='Vitis',
)

hls_model = hls4ml.converters.convert_from_onnx_model(
    onnx_model,
    output_dir='./hls4ml_prjs/hls4ml_prj_t1_v4',
    io_type='io_stream', backend='Vitis',
    hls_config=config, part='xc7z020clg400-1',
)

In [ ]:
if DEBUG:
    hls_model.compile()
    numerical(model=onnx_model, hls_model=hls_model, X=X_test[:2000], plot='boxplot')

In [ ]:
hls_model.compile()
model = t1_1_quantizedCNN(8)
model.load_state_dict(torch.load(pytorch_model_path, map_location=device))
model.eval()
with torch.no_grad():
    y_ref = model(torch.FloatTensor(X_test)).numpy()

y_hls = hls_model.predict(np.ascontiguousarray(X_test, dtype=np.float32))

print('Accuracy (Brevitas float): {}'.format(accuracy_score(y_test, np.argmax(y_ref, axis=1))))
print('Accuracy (HLS): {}'.format(accuracy_score(y_test, np.argmax(y_hls, axis=1))))

In [ ]:
#hls_model.build(csim=False, synth=True)